# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: Logistic Regression

I will use Logistic Regression because my lane is a classification problem: I want to predict trend direction from information available about a content page.

Logistic Regression is a useful first model because it is simple, fast, and interpretable. It gives me a clear ML baseline without adding unnecessary complexity.

I will compare its performance with my Week-4 rule-based baseline using the same data, target, split, and evaluation metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset loaded:", df.shape)

Dataset loaded: (30000, 44)


In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Use the same dataset loaded for the model
model_df = df.copy()

# Target
target = "trend_direction"

# Remove rows without a target
model_df = model_df.dropna(subset=[target])

# Grouped split: pages from the same client stay in one split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nUnique clients:")
print("Training:", train_df["client_id"].nunique())
print("Test:", test_df["client_id"].nunique())

print("\nTarget distribution — training:")
print(train_df[target].value_counts())

print("\nTarget distribution — test:")
print(test_df[target].value_counts())

# Confirm no client appears in both sets
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("\nClient overlap:", len(overlap))

Training rows: 23837
Test rows: 6163

Unique clients:
Training: 25
Test: 7

Target distribution — training:
trend_direction
down      13113
stable     4661
up         3359
new        1934
flat        770
Name: count, dtype: int64

Target distribution — test:
trend_direction
down      3149
stable    1301
up        1029
flat       382
new        302
Name: count, dtype: int64

Client overlap: 0


Why this split is honest

I use an 80/20 grouped split by client so that pages from the same client cannot appear in both training and test data. This reduces the risk that the model benefits from client-specific patterns that would not represent performance on unseen clients.

I use a fixed random seed so the split is reproducible. The Week-4 baseline will be evaluated on this same test set for a fair comparison.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# --------------------------------------------------
# 1. Define target
# --------------------------------------------------

train_df["needs_attention"] = (
    train_df["trend_direction"] == "down"
).astype(int)

test_df["needs_attention"] = (
    test_df["trend_direction"] == "down"
).astype(int)

# --------------------------------------------------
# 2. Features
# --------------------------------------------------

numeric_features = [
    "search_volume",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "ctr"
]

categorical_features = [
    "competition_level"
]

features = numeric_features + categorical_features

X_train = train_df[features]
X_test = test_df[features]

y_train = train_df["needs_attention"]
y_test = test_df["needs_attention"]

# --------------------------------------------------
# 3. Preprocessing
# --------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# --------------------------------------------------
# 4. Logistic Regression
# --------------------------------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

ml_score = model.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 5. Evaluate ML model
# --------------------------------------------------

ml_ap = average_precision_score(y_test, ml_score)
ml_auc = roc_auc_score(y_test, ml_score)

print("Logistic Regression")
print("-------------------")
print("Average Precision:", round(ml_ap, 4))
print("ROC-AUC:", round(ml_auc, 4))

# --------------------------------------------------
# 6. Recreate W04 baseline on SAME test set
# --------------------------------------------------

baseline = test_df.copy()

age_cutoff = df["content_age_days"].median()
ctr_cutoff = df["ctr"].median()
impressions_cutoff = df["impressions_90d"].median()

baseline["baseline_score"] = (
    (baseline["content_age_days"] >= age_cutoff).astype(int)
    + (baseline["ctr"] < ctr_cutoff).astype(int)
    + (baseline["impressions_90d"] >= impressions_cutoff).astype(int)
)

baseline_score = baseline["baseline_score"].astype(float)

baseline_ap = average_precision_score(
    y_test,
    baseline_score
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_score
)

print("\nW04 Baseline")
print("------------")
print("Average Precision:", round(baseline_ap, 4))
print("ROC-AUC:", round(baseline_auc, 4))

# --------------------------------------------------
# 7. Model vs baseline table
# --------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "W04 baseline rule",
        "Logistic Regression"
    ],
    "average_precision": [
        baseline_ap,
        ml_ap
    ],
    "roc_auc": [
        baseline_auc,
        ml_auc
    ]
})

print("\nModel vs Baseline")
print(comparison.to_string(index=False))

Logistic Regression
-------------------
Average Precision: 0.5582
ROC-AUC: 0.5621

W04 Baseline
------------
Average Precision: 0.4974
ROC-AUC: 0.4644

Model vs Baseline
             method  average_precision  roc_auc
  W04 baseline rule           0.497362 0.464380
Logistic Regression           0.558214 0.562052


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# Predictions on the test set
test_results = test_df.copy()

test_results["actual"] = test_results["needs_attention"]
test_results["predicted_probability"] = ml_score
test_results["predicted"] = (ml_score >= 0.5).astype(int)

# Identify incorrect predictions
test_results["correct"] = (
    test_results["actual"] == test_results["predicted"]
)

errors = test_results[~test_results["correct"]].copy()

print("Total test rows:", len(test_results))
print("Incorrect predictions:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(test_results), 4)
)

print("\nExamples of errors:")
print(
    errors[
        [
            "content_id",
            "trend_direction",
            "predicted_probability",
            "content_age_days",
            "ctr",
            "impressions_90d"
        ]
    ].head(10).to_string(index=False)
)

# Feature correlation with the target
print("\nNumeric feature correlations with target:")
print(
    test_results[
        [
            "needs_attention",
            "content_age_days",
            "ctr",
            "impressions_90d",
            "search_volume",
            "cpc"
        ]
    ].corr()["needs_attention"].sort_values()
)

Total test rows: 6163
Incorrect predictions: 2853
Error rate: 0.4629

Examples of errors:
          content_id trend_direction  predicted_probability  content_age_days  ctr  impressions_90d
content_a1fb4e703a9e            down               0.419415               445 0.05            15320
content_a5a2fbc76336          stable               0.527074               238 0.00              307
content_2da6ae9d0882            down               0.368073               502 0.34              297
content_72c5c2d73e5a          stable               0.519708               300 0.12             2426
content_bce275871a25          stable               0.587751               187 1.35              371
content_ff8ea1364b59            down               0.372112               502 0.00              170
content_dcebfd222b10              up               0.662443               145 0.00               16
content_caff51984338            down               0.377585               494 0.00               71
content_68

Error Interpretation

The Logistic Regression model performs better than the Week-4 rule on both Average Precision and ROC-AUC, but its performance is still moderate rather than perfect.

The errors show that the available search and content signals do not completely determine whether a page is trending down. Some pages can have similar age, CTR, and impression patterns but different outcomes.

This suggests that the model captures useful directional patterns, while factors not included in the current feature set may explain some of the remaining errors.

The model is therefore useful as decision support for prioritizing content review, not as proof that a page definitely needs a refresh.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.